# Describe the collected cs.AI corpus

**Milestone 1:** inspect the data, then describe submission volume, categories, and listed author counts over time. Run this notebook yourself, from top to bottom.

Scope: papers tagged cs.AI (primary or additional), first submitted January 2022–October 2026. This is a selected arXiv corpus, not all AI research. Dates describe first submissions; categories and text are the metadata retrieved in November 2026, not historical snapshots. Application papers remain included.

No API calls, text cleaning, dropped records, or file writes. Tables and plots exist only in notebook memory/outputs. Raw data stays unchanged. Unexpected structural problems stop the notebook for review.

Dependencies: pandas, Seaborn, Matplotlib, and an IPython/Jupyter kernel. [Seaborn plotting documentation](https://seaborn.pydata.org/api.html).


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "arxiv_collection.py").is_file():
    project_root = project_root.parent
if not (project_root / "arxiv_collection.py").is_file():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from arxiv_collection import month_ranges

RUN_ID = "2026-11-12_cs-ai"
DATA_DIR = project_root / "data" / "raw" / "arxiv_cs_ai_2022_2026"
TOP_N = 8  # Plotting limit only; full category tables remain available.

PINK = "#D96A9B"
PURPLE = "#7352A2"
GREEN = "#39866A"
COLORS = [PURPLE, PINK, GREEN, "#B48AC7", "#B9457C", "#76B89A", "#9C4F96", "#E79BB8", "#245D4A"]
sns.set_theme(style="whitegrid", context="notebook", palette=COLORS, rc={
    "figure.figsize": (12, 4.5), "axes.facecolor": "#FCF9FC",
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.alpha": 0.25, "axes.titleweight": "bold",
})
print({"Python": platform.python_version(), "pandas": pd.__version__,
       "seaborn": sns.__version__, "matplotlib": matplotlib.__version__})


## 1. Load exactly the recorded collection

Read the run record and its expected months. Check monthly settings, data hashes, and saved counts before loading. This avoids silently accepting missing months or mixing runs. It checks consistency with the collection records, not completeness of arXiv itself.


In [ ]:
run = json.loads((project_root / "collection_runs" / f"{RUN_ID}.json").read_text(encoding="utf-8"))
if run["retrieval_finished"] is None or run["total_records"] is None:
    raise RuntimeError("The selected run is not recorded as complete.")
start_bound = pd.Timestamp(run["submission_start"])
end_exclusive = pd.Timestamp(run["submission_end"]) + pd.Timedelta(seconds=1)
windows = month_ranges(start_bound.strftime("%Y-%m"), pd.Timestamp(run["submission_end"]).strftime("%Y-%m"))

all_records = []
file_rows = []
for start_date, end_date in windows:
    stem = f"{start_date}_{end_date}"
    manifest_path = DATA_DIR / f"{stem}.manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected = {"categories": run["categories"], "start_date": start_date, "end_date": end_date}
    if manifest["status"] != "complete" or manifest["settings"] != expected:
        raise RuntimeError(f"Unexpected collection settings: {manifest_path.name}")
    payload = (DATA_DIR / manifest["data_file"]).read_bytes()
    if hashlib.sha256(payload).hexdigest() != manifest["sha256"]:
        raise RuntimeError(f"Data hash mismatch: {manifest['data_file']}")
    records = [json.loads(line) for line in payload.decode("utf-8").splitlines()]
    if len(records) != manifest["retrieved_count"] or len(records) != manifest["api_total"]:
        raise RuntimeError(f"Count mismatch: {manifest['data_file']}")
    all_records.extend(records)
    file_rows.append({"month": start_date[:6], "records": len(records)})

df = pd.DataFrame(all_records)
del all_records, records, payload
if len(df) != run["total_records"]:
    raise RuntimeError("Loaded count differs from the run record.")
required = ["arxiv_id", "title", "abstract", "submitted_at", "updated_at", "primary_category", "categories", "authors"]
if not set(required).issubset(df.columns):
    raise RuntimeError("Expected fields are missing from the dataset.")
print("Run:", RUN_ID)
print("Monthly files:", len(file_rows))
print("Loaded records:", f"{len(df):,}")
print("Submission period:", run["submission_start"], "through", run["submission_end"])
display(df[["arxiv_id", "title", "submitted_at", "primary_category", "categories", "authors"]].head())


## 2. Inspect quality before aggregation

Keep text unchanged. Parse dates into temporary series and check author/category list structure. Missing text is reported for review. Invalid identifiers, submission dates, or category/author structure stop the plots rather than silently excluding records. Repeated author names are flagged, but names are not merged or deduplicated.


In [ ]:
blank = pd.DataFrame({
    "field": ["arxiv_id", "title", "abstract", "submitted_at", "updated_at", "primary_category"],
    "missing_or_blank": [
        int(df[column].map(lambda value: not isinstance(value, str) or not value.strip()).sum())
        for column in ["arxiv_id", "title", "abstract", "submitted_at", "updated_at", "primary_category"]
    ],
})
display(blank)

submitted = pd.to_datetime(df["submitted_at"], utc=True, errors="coerce", format="ISO8601")
updated = pd.to_datetime(df["updated_at"], utc=True, errors="coerce", format="ISO8601")
valid_ids = df["arxiv_id"].map(lambda value: isinstance(value, str) and bool(value.strip()))
base_ids = df["arxiv_id"].where(valid_ids, "").str.replace(r"v\d+$", "", regex=True)
valid_authors = df["authors"].map(lambda values: isinstance(values, list) and len(values) > 0 and all(isinstance(v, str) and bool(v.strip()) for v in values))
valid_categories = df["categories"].map(lambda values: isinstance(values, list) and len(values) > 0 and all(isinstance(v, str) and bool(v.strip()) for v in values))
valid_primary = df["primary_category"].map(lambda value: isinstance(value, str) and bool(value.strip()))
in_scope = (submitted >= start_bound) & (submitted < end_exclusive)
category_membership = pd.Series([
    bool(set(values).intersection(run["categories"])) if valid else False
    for values, valid in zip(df["categories"], valid_categories)
], index=df.index)
primary_in_categories = pd.Series([
    primary in values if valid else False
    for primary, values, valid in zip(df["primary_category"], df["categories"], valid_categories)
], index=df.index)
repeated_categories = pd.Series([
    len(values) != len(set(values)) if valid else False
    for values, valid in zip(df["categories"], valid_categories)
], index=df.index)
repeated_author_names = pd.Series([
    len(values) != len(set(values)) if valid else False
    for values, valid in zip(df["authors"], valid_authors)
], index=df.index)

blocking = {
    "missing_or_invalid_ids": int((~valid_ids).sum()),
    "duplicate_ids_ignoring_versions": int(base_ids[valid_ids].duplicated().sum()),
    "invalid_submission_dates": int(submitted.isna().sum()),
    "outside_submission_range": int((~in_scope & submitted.notna()).sum()),
    "invalid_or_empty_author_lists": int((~valid_authors).sum()),
    "invalid_or_empty_category_lists": int((~valid_categories).sum()),
    "missing_primary_category": int((~valid_primary).sum()),
    "primary_not_in_category_list": int((~primary_in_categories).sum()),
    "without_requested_category": int((~category_membership).sum()),
    "repeated_categories_within_record": int(repeated_categories.sum()),
}
display(pd.Series(blocking, name="records_flagged").to_frame())
print("Records with repeated author names:", int(repeated_author_names.sum()))
print("Missing or invalid revision dates:", int(updated.isna().sum()))
print("Revision dates earlier than submission:", int((updated < submitted).sum()))
print("Revision dates after the collection's submission cutoff:", int((updated >= end_exclusive).sum()))
if any(blocking.values()):
    display(df.loc[~valid_authors | ~valid_categories | ~in_scope, ["arxiv_id", "submitted_at", "authors", "categories"]].head(10))
    raise RuntimeError("Review the flagged issues and agree on handling before plotting. No records were removed.")


## 3. Derive descriptive fields in memory

The analysis table adds submission month/year, listed author count, and whether cs.AI is the primary category. Counting author entries does not identify unique people, resolve name variants, or measure unique researchers.


In [ ]:
analysis = df.copy()
analysis["submitted"] = submitted
analysis["month"] = submitted.dt.tz_localize(None).dt.to_period("M").dt.to_timestamp()
analysis["year"] = submitted.dt.year
analysis["author_count"] = analysis["authors"].map(len)
analysis["cs_ai_role"] = analysis["primary_category"].eq("cs.AI").map({True: "Primary cs.AI", False: "Cross-listed cs.AI"})
analysis["companion_categories"] = [
    [category for category in values if category != "cs.AI"]
    for values in analysis["categories"]
]
all_months = pd.date_range(start_bound.tz_localize(None).replace(day=1), pd.Timestamp(run["submission_end"]).tz_localize(None).replace(day=1, hour=0, minute=0, second=0), freq="MS", name="month")
monthly_volume = analysis.groupby("month").size().reindex(all_months, fill_value=0).rename("papers")
monthly_table = monthly_volume.reset_index()
print("Submission dates:", submitted.min(), "to", submitted.max())
print("Listed authors per paper:")
display(analysis["author_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).to_frame())
display(analysis["cs_ai_role"].value_counts().to_frame("papers"))


## 4. Submission volume over time

Each paper is counted once, in its first-submission month. These are publication-activity counts, not research quality or impact. November–December 2026 are outside collection, not zero-submission months.


In [ ]:
fig, ax = plt.subplots()
sns.lineplot(data=monthly_table, x="month", y="papers", color=PURPLE, marker="o", estimator=None, ax=ax)
ax.set(title="Monthly submissions in the collected cs.AI corpus", xlabel="First-submission month (UTC)", ylabel="Papers")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Compare equal January–October exposure for every year, without annualizing 2026.
year_summary = analysis.groupby("year").agg(papers=("arxiv_id", "size"), observed_months=("month", "nunique"))
jan_oct = analysis.loc[analysis["submitted"].dt.month <= 10].groupby("year").size()
year_summary["january_october_papers"] = jan_oct.reindex(year_summary.index, fill_value=0)
display(year_summary)


## 5. Primary cs.AI versus cross-listed cs.AI

These groups are mutually exclusive. Monthly shares use **all collected papers in that month** as the denominator and sum to 100% when the month has records. This comparison can inform a later scope decision; it does not exclude either group now.


In [ ]:
roles = ["Primary cs.AI", "Cross-listed cs.AI"]
role_counts = analysis.groupby(["month", "cs_ai_role"]).size().unstack(fill_value=0).reindex(index=all_months, columns=roles, fill_value=0)
role_shares = role_counts.div(monthly_volume.replace(0, float("nan")), axis=0) * 100
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for table, ax, ylabel in [(role_counts, axes[0], "Papers"), (role_shares, axes[1], "Share of monthly corpus (%)")]:
    long = table.rename_axis(columns="role").reset_index().melt(id_vars="month", var_name="role", value_name="value")
    sns.lineplot(data=long, x="month", y="value", hue="role", palette={roles[0]: PURPLE, roles[1]: GREEN}, estimator=None, ax=ax)
    ax.set(xlabel="Submission month", ylabel=ylabel)
    ax.tick_params(axis="x", rotation=45)
axes[0].set_title("Submissions by cs.AI role")
axes[1].set(title="Monthly share by cs.AI role", ylim=(0, 100))
plt.tight_layout()
plt.show()


## 6. Primary categories and other categories alongside cs.AI

Two views:

- **Primary categories:** count each paper once under its primary category.
- **Other categories appearing alongside cs.AI:** count papers carrying each non-cs.AI tag, whether that tag is primary or additional.

Example: a paper tagged `[cs.LG, cs.AI, stat.ML]` with primary `cs.LG` contributes once to `cs.LG` in the first view, and once each to `cs.LG` and `stat.ML` in the second. The second view excludes cs.AI because every collected paper already carries it.

Both monthly shares divide by **all collected papers in that month**, including papers with no other tags. Other-category shares can overlap and need not sum to 100%. Robotics here means robotics papers also tagged cs.AI, not all robotics papers. Primary category does not establish whether a paper advances AI methods or applies them.

Full monthly tables are available as `primary_counts` and `companion_counts`. Charts show the eight largest categories in each view across the full period; smaller categories remain in the tables and data. Displayed primary-category shares may therefore sum to less than 100%.


In [ ]:
primary_counts = analysis.groupby(["month", "primary_category"]).size().unstack(fill_value=0).reindex(all_months, fill_value=0)
companion_rows = analysis[["month", "companion_categories"]].explode("companion_categories")
companion_counts = companion_rows.groupby(["month", "companion_categories"]).size().unstack(fill_value=0).reindex(all_months, fill_value=0)
category_summary = pd.DataFrame({
    "papers_with_primary_category": primary_counts.sum(),
    "papers_with_this_tag_alongside_cs_ai": companion_counts.sum(),
}).fillna(0).astype(int).sort_values("papers_with_this_tag_alongside_cs_ai", ascending=False)
display(category_summary)
category_order = list(dict.fromkeys(
    primary_counts.sum().sort_values(ascending=False).index.tolist()
    + companion_counts.sum().sort_values(ascending=False).index.tolist()
))
category_colors = {category: COLORS[i % len(COLORS)] for i, category in enumerate(category_order)}

# Use the same colors and denominator in both category views.
def plot_category_trends(counts, title):
    top = counts.sum().nlargest(TOP_N).index.tolist()
    if not top:
        print("No categories to plot:", title)
        return
    selected = counts[top]
    shares = selected.div(monthly_volume.replace(0, float("nan")), axis=0) * 100
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for table, ax, ylabel in [(selected, axes[0], "Papers"), (shares, axes[1], "Share of monthly corpus (%)")]:
        long = table.rename_axis(columns="category").reset_index().melt(id_vars="month", var_name="category", value_name="value")
        sns.lineplot(data=long, x="month", y="value", hue="category", hue_order=top, palette=category_colors, estimator=None, ax=ax)
        ax.set(xlabel="Submission month", ylabel=ylabel)
        ax.tick_params(axis="x", rotation=45)
        ax.legend(fontsize=8, title="Category", ncol=2)
    axes[0].set_title(f"{title}: top {len(top)} by volume")
    axes[1].set(title=f"{title}: monthly shares", ylim=(0, 100))
    plt.tight_layout()
    plt.show()


In [ ]:
plot_category_trends(primary_counts, "Primary categories")
plot_category_trends(companion_counts, "Other categories alongside cs.AI")


### Primary cs.AI versus primary cs.LG

Compare monthly submission counts and monthly shares for these two primary categories. Shares use **all papers in the collected corpus that month** as the denominator, not just these two groups, so the two lines need not sum to 100%.

cs.LG means machine learning. Every paper here is tagged cs.AI somewhere: the primary cs.LG group therefore contains only machine-learning papers also tagged cs.AI. This is not a comparison of all cs.AI and all cs.LG submissions on arXiv. Changes can reflect category choices as well as research content; they do not alone establish a shift away from machine learning.


In [ ]:
comparison_categories = ["cs.AI", "cs.LG"]
ai_lg_counts = primary_counts.reindex(columns=comparison_categories, fill_value=0)
ai_lg_shares = ai_lg_counts.div(monthly_volume.replace(0, float("nan")), axis=0) * 100
ai_lg_colors = {"cs.AI": PURPLE, "cs.LG": GREEN}

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for table, ax, ylabel in [
    (ai_lg_counts, axes[0], "Papers"),
    (ai_lg_shares, axes[1], "Share of all collected papers that month (%)"),
]:
    long = table.rename_axis(columns="primary_category").reset_index().melt(
        id_vars="month", var_name="primary_category", value_name="value"
    )
    sns.lineplot(
        data=long, x="month", y="value", hue="primary_category",
        hue_order=comparison_categories, palette=ai_lg_colors,
        estimator=None, ax=ax,
    )
    ax.set(xlabel="First-submission month", ylabel=ylabel)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="Primary category")
axes[0].set_title("Primary cs.AI versus cs.LG: monthly submissions")
axes[1].set(title="Primary cs.AI versus cs.LG: monthly shares", ylim=(0, 100))
plt.tight_layout()
plt.show()


## 7. Listed author counts by year

Each yearly box shows the median and middle 50% of listed author counts. Whiskers extend to the most extreme observations within 1.5 times the interquartile range of the box; points beyond them are shown. These points are not automatically errors. All papers remain included.

The author-count axis is logarithmic to keep both small teams and the long tail visible. Box statistics are calculated from the original author counts. The 2026 box covers January–October; earlier boxes cover complete years, so seasonal differences could affect comparisons.

Monthly and yearly summary tables remain available. Counts describe author entries, not resolved individual identities or unique researchers.


In [ ]:
author_monthly = analysis.groupby("month")["author_count"].agg(
    papers="size", median="median", q1=lambda values: values.quantile(0.25), q3=lambda values: values.quantile(0.75)
).reindex(all_months)
author_yearly = analysis.groupby("year")["author_count"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
display(author_yearly)
display(analysis.groupby("cs_ai_role")["author_count"].describe())


In [ ]:
from matplotlib.ticker import ScalarFormatter

years = sorted(analysis["year"].unique())
year_colors = {year: COLORS[i % len(COLORS)] for i, year in enumerate(years)}
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=analysis, x="year", y="author_count", hue="year", order=years,
    hue_order=years, palette=year_colors, dodge=False, legend=False,
    showfliers=True, fliersize=1.5, linewidth=1, ax=ax,
)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.set_xticks(range(len(years)), labels=[
    f"{year} (Jan–Oct)" if year == 2026 else str(year) for year in years
])
ax.set(title="Listed authors per paper by submission year", xlabel="Submission year", ylabel="Listed author count (log scale)")
plt.tight_layout()
plt.show()


## Review before cleaning or text analysis

Record your observations here after running the notebook:

- Which quality flags need a handling rule? Do unusual author lists or counts reflect metadata issues or real collaborations?
- How do primary cs.AI and cross-listed papers differ? Should later analysis retain both?
- Which category changes appear in counts, shares, or both? Could changes in cross-listing practices or corpus selection explain them?
- Do author-count changes persist when comparing similar categories and equal calendar periods?

These are descriptive results. Publication counts do not measure quality or establish research gaps. Agree on any exclusions, cleaning, or narrower scope before changing the data. Text methods and AI safety classification remain later milestones.
